In [17]:
# ===== Cell 1: Install & imports =====
!pip -q install pytorchvideo decord opencv-contrib-python scikit-learn tqdm

import os
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report, confusion_matrix

import cv2
import decord
from decord import VideoReader, cpu

from pytorchvideo.models.hub import x3d_s
from torchvision import models

In [18]:
# ===== Cell 2: Config, paths, constants =====
# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# === PATHS ===
# For your local Windows machine:
ROOT = Path(r"E:\Graduation Project")

# If using Colab instead, comment the above and use:
# ROOT = Path("/content/drive/MyDrive/Graduation Project")

EVAL_DIR   = ROOT / "eval_data"      # contains non_violence / violence
MODELS_DIR = ROOT / "models"         # where x3d_s_best.pth & mobilenet_clip_best.pth live

print("Eval dir:", EVAL_DIR)
print("Models dir:", MODELS_DIR)

# === Model / data constants ===
NUM_CLASSES = 2

# X3D and MobileNet were both trained with T=13, SIZE=160 in your last runs
T_X3D   = 13
T_MNET  = 13
SIZE    = 160

# Normalization (ImageNet)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Thresholds chosen from your test sweeps
THRESH_X3D  = 0.60   # from X3D test sweep
THRESH_MNET = 0.55   # from MobileNet test sweep

# Decord setup
decord.bridge.set_bridge("native")

Using device: cuda
Eval dir: E:\Graduation Project\eval_data
Models dir: E:\Graduation Project\models


In [19]:
# ===== Cell 3: Shared eval video listing helper =====
def list_eval_videos(root_dir: Path):
    """
    Scans:
      root_dir/non_violence/*.mp4 -> label 0
      root_dir/violence/*.mp4     -> label 1

    Returns: list of (path, label) that are readable.
    """
    candidates = []
    for label_name, label_id in [("non_violence", 0), ("violence", 1)]:
        folder = root_dir / label_name
        files = sorted(folder.glob("*.mp4"))
        print(f"Found {len(files)} videos in {folder}")
        for p in files:
            candidates.append((str(p), label_id))

    valid = []
    for path, label in candidates:
        try:
            vr = VideoReader(path, ctx=cpu(0))
            if len(vr) == 0:
                raise RuntimeError("no frames")
            valid.append((path, label))
        except Exception as e:
            print(f"[WARN] Cannot open {path}: {e} -> skipping")

    print(f"Total valid eval videos: {len(valid)}")
    return valid

In [20]:
# ===== Cell 4: EvalDataset for X3D ([C, T, H, W]) =====
class EvalDatasetX3D(Dataset):
    """
    For X3D-S:
      - returns (C, T, H, W) float32 tensor
      - labels: 0 = NonViolence, 1 = Violence
    """
    def __init__(self, root_dir: Path, clip_len=T_X3D, frame_size=SIZE):
        self.samples = list_eval_videos(root_dir)
        self.clip_len = clip_len
        self.frame_size = frame_size

    def __len__(self):
        return len(self.samples)

    def _indices(self, n_frames: int):
        if n_frames <= self.clip_len:
            return np.linspace(0, n_frames - 1, self.clip_len).astype(int)
        # center clip (deterministic for eval)
        start = (n_frames - self.clip_len) // 2
        return np.arange(start, start + self.clip_len)

    def _resize_center(self, img: np.ndarray):
        h, w = img.shape[:2]
        scale = int(self.frame_size * 1.14)
        short = min(h, w)
        r = scale / float(short)
        img = cv2.resize(img, (int(w * r), int(h * r)))
        h, w = img.shape[:2]

        y = max((h - self.frame_size) // 2, 0)
        x = max((w - self.frame_size) // 2, 0)
        img = img[y:y + self.frame_size, x:x + self.frame_size]
        return img

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        vr = VideoReader(path, ctx=cpu(0))
        idxs = self._indices(len(vr))
        frames = vr.get_batch(idxs).asnumpy()  # (T, H, W, 3) uint8

        frames = [self._resize_center(f) for f in frames]
        frames = np.stack(frames).astype(np.float32) / 255.0  # (T, H, W, 3)
        frames = (frames - IMAGENET_MEAN) / IMAGENET_STD
        frames = np.transpose(frames, (3, 0, 1, 2))           # (C, T, H, W)

        return torch.from_numpy(frames), torch.tensor(label, dtype=torch.long)

eval_ds_x3d = EvalDatasetX3D(EVAL_DIR, clip_len=T_X3D, frame_size=SIZE)
eval_loader_x3d = DataLoader(
    eval_ds_x3d,
    batch_size=2,
    shuffle=False,
    num_workers=0,   # safer on Windows
    pin_memory=True
)

Found 60 videos in E:\Graduation Project\eval_data\non_violence
Found 60 videos in E:\Graduation Project\eval_data\violence
Total valid eval videos: 120


In [21]:
# ===== Cell 5: Load X3D-S best model & evaluate =====
# Rebuild X3D-S head to match NUM_CLASSES, then load your best weights
model_x3d = x3d_s(pretrained=False)
in_features = model_x3d.blocks[5].proj.in_features
model_x3d.blocks[5].proj = nn.Linear(in_features, NUM_CLASSES)

state_x3d = torch.load(MODELS_DIR / "x3d_s_best.pth", map_location=DEVICE)
model_x3d.load_state_dict(state_x3d)
model_x3d = model_x3d.to(DEVICE)
model_x3d.eval()

all_probs_x3d = []
all_labels_x3d = []

with torch.no_grad():
    for clips, labels in eval_loader_x3d:
        # clips: [B, C, T, H, W]
        clips  = clips.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits = model_x3d(clips)
        probs  = torch.softmax(logits, dim=1)[:, 1]  # P(violence)

        all_probs_x3d.extend(probs.cpu().numpy().tolist())
        all_labels_x3d.extend(labels.cpu().numpy().tolist())

all_probs_x3d  = np.array(all_probs_x3d)
all_labels_x3d = np.array(all_labels_x3d)

preds_x3d = (all_probs_x3d >= THRESH_X3D).astype(int)

print(f"\n=== X3D-S on eval_data (threshold={THRESH_X3D}) ===")
print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(all_labels_x3d, preds_x3d))

print("\nClassification report:")
print(classification_report(
    all_labels_x3d,
    preds_x3d,
    target_names=["NonViolence", "Violence"],
    digits=4
))

C:\Users\abdal\AppData\Local\Temp\ipykernel_41708\3293330630.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_x3d = torch.load(MODELS_DIR / "x3d_s_best.pth", map_lo


=== X3D-S on eval_data (threshold=0.6) ===
Confusion matrix [rows=true, cols=pred]:
[[52  8]
 [57  3]]

Classification report:
              precision    recall  f1-score   support

 NonViolence     0.4771    0.8667    0.6154        60
    Violence     0.2727    0.0500    0.0845        60

    accuracy                         0.4583       120
   macro avg     0.3749    0.4583    0.3499       120
weighted avg     0.3749    0.4583    0.3499       120



In [22]:
# ===== Cell 6: EvalDataset for MobileNetClip ([T, C, H, W]) =====
class EvalDatasetMobileNet(Dataset):
    """
    For MobileNetClip:
      - returns (T, C, H, W)
      - labels: 0 = NonViolence, 1 = Violence
    """
    def __init__(self, root_dir: Path, clip_len=T_MNET, frame_size=SIZE):
        self.items = list_eval_videos(root_dir)
        self.clip_len = clip_len
        self.frame_size = frame_size

    def __len__(self):
        return len(self.items)

    def _indices(self, n_frames: int):
        if n_frames <= self.clip_len:
            return np.linspace(0, n_frames - 1, self.clip_len).astype(int)
        start = (n_frames - self.clip_len) // 2
        return np.arange(start, start + self.clip_len)

    def _resize_center(self, img: np.ndarray):
        h, w = img.shape[:2]
        scale = int(self.frame_size * 1.14)
        short = min(h, w)
        r = scale / float(short)
        img = cv2.resize(img, (int(w * r), int(h * r)))
        h, w = img.shape[:2]

        y = max((h - self.frame_size) // 2, 0)
        x = max((w - self.frame_size) // 2, 0)
        img = img[y:y + self.frame_size, x:x + self.frame_size]
        return img

    def __getitem__(self, idx):
        path, label = self.items[idx]
        vr = VideoReader(path, ctx=cpu(0))
        idxs = self._indices(len(vr))
        frames = vr.get_batch(idxs).asnumpy()  # (T, H, W, 3)

        frames = [self._resize_center(f) for f in frames]
        frames = np.stack(frames).astype(np.float32) / 255.0  # (T, H, W, 3)
        frames = (frames - IMAGENET_MEAN) / IMAGENET_STD
        frames = np.transpose(frames, (0, 3, 1, 2))           # (T, C, H, W)

        return torch.from_numpy(frames), torch.tensor(label, dtype=torch.long)

eval_ds_mnet = EvalDatasetMobileNet(EVAL_DIR, clip_len=T_MNET, frame_size=SIZE)
eval_loader_mnet = DataLoader(
    eval_ds_mnet,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

Found 60 videos in E:\Graduation Project\eval_data\non_violence
Found 60 videos in E:\Graduation Project\eval_data\violence
Total valid eval videos: 120


In [23]:
# ===== Cell 7: Define MobileNetClip, load best, evaluate =====
class MobileNetClip(nn.Module):
    """
    Wraps MobileNetV2 to handle [B, T, C, H, W]:
      - Flatten frames into batch
      - Run MobileNet on each frame
      - Average logits across T
    """
    def __init__(self, backbone, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = backbone
        # make sure classifier has correct output size
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # x: [B, T, C, H, W]
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        logits_frame = self.backbone(x)         # [B*T, num_classes]
        logits_frame = logits_frame.view(B, T, NUM_CLASSES)
        logits = logits_frame.mean(dim=1)       # [B, num_classes]
        return logits

# Rebuild backbone (weights=None is fine, we'll overwrite with our trained weights)
backbone = models.mobilenet_v2(weights=None)
model_mnet = MobileNetClip(backbone, num_classes=NUM_CLASSES).to(DEVICE)

state_mnet = torch.load(MODELS_DIR / "mobilenet_clip_best.pth", map_location=DEVICE)
model_mnet.load_state_dict(state_mnet)
model_mnet.eval()

all_probs_mnet = []
all_labels_mnet = []

with torch.no_grad():
    for frames, labels in eval_loader_mnet:
        # frames: [B, T, C, H, W]
        frames = frames.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits = model_mnet(frames)
        probs  = torch.softmax(logits, dim=1)[:, 1]  # P(violence)

        all_probs_mnet.extend(probs.cpu().numpy().tolist())
        all_labels_mnet.extend(labels.cpu().numpy().tolist())

all_probs_mnet  = np.array(all_probs_mnet)
all_labels_mnet = np.array(all_labels_mnet)

preds_mnet = (all_probs_mnet >= THRESH_MNET).astype(int)

print(f"\n=== MobileNetClip on eval_data (threshold={THRESH_MNET}) ===")
print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(all_labels_mnet, preds_mnet))

print("\nClassification report:")
print(classification_report(
    all_labels_mnet,
    preds_mnet,
    target_names=["NonViolence", "Violence"],
    digits=4
))

C:\Users\abdal\AppData\Local\Temp\ipykernel_41708\3274067344.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_mnet = torch.load(MODELS_DIR / "mobilenet_clip_best.p


=== MobileNetClip on eval_data (threshold=0.55) ===
Confusion matrix [rows=true, cols=pred]:
[[26 34]
 [29 31]]

Classification report:
              precision    recall  f1-score   support

 NonViolence     0.4727    0.4333    0.4522        60
    Violence     0.4769    0.5167    0.4960        60

    accuracy                         0.4750       120
   macro avg     0.4748    0.4750    0.4741       120
weighted avg     0.4748    0.4750    0.4741       120



In [24]:
from sklearn.metrics import classification_report

def eval_x3d_at_threshold(th):
    preds = (all_probs_x3d >= th).astype(int)
    rep = classification_report(
        all_labels_x3d,
        preds,
        target_names=["NonViolence", "Violence"],
        digits=4,
        output_dict=True
    )
    return {
        "threshold": float(th),
        "violence_precision": rep["Violence"]["precision"],
        "violence_recall":    rep["Violence"]["recall"],
        "violence_f1":        rep["Violence"]["f1-score"],
        "accuracy":           rep["accuracy"],
    }

ths = np.linspace(0.1, 0.9, 17)
results_x3d = [eval_x3d_at_threshold(t) for t in ths]

print("X3D threshold sweep on eval_data:")
for r in results_x3d:
    print(r)

best_x3d = max(results_x3d, key=lambda x: x["violence_f1"])
print("\nBest by F1:", best_x3d)

X3D threshold sweep on eval_data:
{'threshold': 0.1, 'violence_precision': 0.5, 'violence_recall': 1.0, 'violence_f1': 0.6666666666666666, 'accuracy': 0.5}
{'threshold': 0.15000000000000002, 'violence_precision': 0.5, 'violence_recall': 1.0, 'violence_f1': 0.6666666666666666, 'accuracy': 0.5}
{'threshold': 0.2, 'violence_precision': 0.5, 'violence_recall': 1.0, 'violence_f1': 0.6666666666666666, 'accuracy': 0.5}
{'threshold': 0.25, 'violence_precision': 0.5, 'violence_recall': 1.0, 'violence_f1': 0.6666666666666666, 'accuracy': 0.5}
{'threshold': 0.30000000000000004, 'violence_precision': 0.5079365079365079, 'violence_recall': 0.5333333333333333, 'violence_f1': 0.5203252032520326, 'accuracy': 0.5083333333333333}
{'threshold': 0.35, 'violence_precision': 0.5106382978723404, 'violence_recall': 0.4, 'violence_f1': 0.4485981308411215, 'accuracy': 0.5083333333333333}
{'threshold': 0.4, 'violence_precision': 0.5, 'violence_recall': 0.26666666666666666, 'violence_f1': 0.34782608695652173, 'ac

C:\Users\abdal\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\abdal\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\abdal\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [25]:
def eval_mnet_at_threshold(th):
    preds = (all_probs_mnet >= th).astype(int)
    rep = classification_report(
        all_labels_mnet,
        preds,
        target_names=["NonViolence", "Violence"],
        digits=4,
        output_dict=True
    )
    return {
        "threshold": float(th),
        "violence_precision": rep["Violence"]["precision"],
        "violence_recall":    rep["Violence"]["recall"],
        "violence_f1":        rep["Violence"]["f1-score"],
        "accuracy":           rep["accuracy"],
    }

ths = np.linspace(0.1, 0.9, 17)
results_mnet = [eval_mnet_at_threshold(t) for t in ths]

print("MobileNet threshold sweep on eval_data:")
for r in results_mnet:
    print(r)

best_mnet = max(results_mnet, key=lambda x: x["violence_f1"])
print("\nBest by F1:", best_mnet)

MobileNet threshold sweep on eval_data:
{'threshold': 0.1, 'violence_precision': 0.5092592592592593, 'violence_recall': 0.9166666666666666, 'violence_f1': 0.6547619047619048, 'accuracy': 0.5166666666666667}
{'threshold': 0.15000000000000002, 'violence_precision': 0.5, 'violence_recall': 0.8333333333333334, 'violence_f1': 0.625, 'accuracy': 0.5}
{'threshold': 0.2, 'violence_precision': 0.4842105263157895, 'violence_recall': 0.7666666666666667, 'violence_f1': 0.5935483870967742, 'accuracy': 0.475}
{'threshold': 0.25, 'violence_precision': 0.4777777777777778, 'violence_recall': 0.7166666666666667, 'violence_f1': 0.5733333333333334, 'accuracy': 0.4666666666666667}
{'threshold': 0.30000000000000004, 'violence_precision': 0.46511627906976744, 'violence_recall': 0.6666666666666666, 'violence_f1': 0.547945205479452, 'accuracy': 0.45}
{'threshold': 0.35, 'violence_precision': 0.4691358024691358, 'violence_recall': 0.6333333333333333, 'violence_f1': 0.5390070921985816, 'accuracy': 0.458333333333